In [0]:
%run ./00_config

In [0]:
%run ./02_core_functions

# 03_mainOrchestrates 
The full pipeline for every active control row: **copy** the zip files from `zip_source_path` to their date-partitioned `zip_tgt_path`, then **unzip** only the files that were copied successfully in this run into `unzip_tgt_path`.

### Run setup

In [0]:
run_id = str(uuid.uuid4())
control_entries = get_active_control_entries()

print(f"run_id = {run_id}")
print(f"Processing {len(control_entries)} active control row(s)")

all_log_records = []

### Stage 1 — Copy
For every active control row, list zip files at `zip_source_path`, then keep only the ones that
should be processed this run:
- `is_full_load = 1` -> all zip files found are copied (unchanged behavior).
- `is_incremental_load = 1` -> only files whose filename date token matches **today's date**
  (`yyyyMMdd`, system date) are copied. If none match, a `FAILED` log record is written for that
  control row and no files are copied for it.

Selected files are copied into the matching `yyyyMMdd` folder under `zip_tgt_path`.

In [0]:
for ctl in control_entries:
    print(f"--- s_no={ctl['s_no']} | source={ctl['zip_source']} | {ctl['zip_source_path']} ---")

    all_zip_files = list_zip_files(ctl["zip_source_path"])
    zip_files = filter_zip_files_by_load_type(
        all_zip_files,
        is_full_load=ctl["is_full_load"],
        is_incremental_load=ctl["is_incremental_load"],
    )
    print(f"Found {len(all_zip_files)} zip file(s) at source, {len(zip_files)} selected for processing "
          f"(is_full_load={ctl['is_full_load']}, is_incremental_load={ctl['is_incremental_load']})")

    if ctl["is_incremental_load"] == 1 and len(zip_files) == 0:
        today_token = datetime.now().strftime("%Y%m%d")
        now = datetime.now()
        failure_record = {
            "s_no": ctl["s_no"],
            "stage": "COPY",
            "source": ctl["zip_source"],
            "file_name": None,
            "source_path": ctl["zip_source_path"],
            "target_path": None,
            "status": "FAILED",
            "message": f"Incremental load: no zip file found for today's date {today_token}",
            "start_time": now,
            "end_time": now,
        }
        all_log_records.append(failure_record)
        print(f"  [COPY:FAILED] {failure_record['message']}")
        continue

    for file_info in zip_files:
        record = copy_zip_file(
            file_info=file_info,
            target_base_path=ctl["zip_tgt_path"],
            s_no=ctl["s_no"],
            source=ctl["zip_source"],
        )
        all_log_records.append(record)
        print(f"  [COPY:{record['status']}] {record['file_name']}: {record['message']}")

### Stage 2 — UnzipOnly the files that **copied successfully in this run** are unzipped, into the `unzip_tgt_path` for their control row. Files copied in earlier runs are not re-touched.

In [0]:
# s_no -> unzip_tgt_path, for routing each copied file to the right destination
unzip_targets = {ctl["s_no"]: ctl["unzip_tgt_path"] for ctl in control_entries}

copied_this_run = [
    r for r in all_log_records
    if r["stage"] == "COPY" and r["status"] == "SUCCESS"
]
print(f"{len(copied_this_run)} file(s) copied successfully this run — proceeding to unzip")

for copy_record in copied_this_run:
    unzip_base = unzip_targets.get(copy_record["s_no"])
    if not unzip_base:
        print(f"  [UNZIP:SKIPPED] {copy_record['file_name']}: no unzip_tgt_path for s_no={copy_record['s_no']}")
        continue

    unzip_record = unzip_zip_file(
        copy_record=copy_record,
        unzip_base_path=unzip_base,
    )
    all_log_records.append(unzip_record)
    print(f"  [UNZIP:{unzip_record['status']}] {unzip_record['file_name']}: {unzip_record['message']}")

### Write Logs 
Both `COPY` and `UNZIP` records for this `run_id` are written in the same log table.

In [0]:
write_logs(all_log_records, run_id)
print(f"Logged {len(all_log_records)} record(s) to {log_table}")

### Run summary

In [0]:
%sql
SELECT * FROM iics.casemix.zip_files_execution_log_test
ORDER BY start_time